In [ ]:
from collections import Counter

import itertools
import stim
from IPython.core.display import Markdown

from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch

In [ ]:
instructions = Counter()
def rewrite_with_polygons(filename: str, steane: SteaneCodePatch):
    # Insert all the polygons into the Stim file for readability.
    with open(filename, "r", encoding="utf-8") as file:
        lines = file.readlines()
        inserted = 0

        # Insert polygons for the initial stabilizers of the Steane Code patch
        for polygon in steane.get_initial_polygons():
            lines.insert(instructions['preparation'] + inserted, polygon)
            inserted += 1
        lines.insert(instructions['preparation'] + inserted, "TICK\n")
        inserted += 1

        # Insert polygons for the prepared Steane Code patch prior to Superdense Syndrome Measurement
        for polygon in steane.get_prepared_polygons():
            lines.insert(instructions['superdense'] + inserted, polygon)
            inserted += 1
        lines.insert(instructions['superdense'] + inserted, "TICK\n")
        inserted += 1

    with open(filename, "w", encoding="utf-8") as file:
        file.writelines(lines)
        print(f"Generated circuit : {filename}")

In [ ]:
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions = (5, 3))
steane = SteaneCodePatch(array)

instructions['preparation'] = len(circuit)
steane.append_preparation(circuit)
instructions['superdense'] = len(circuit)
steane.append_superdense(circuit, prefix="SDC0")
steane.append_superdense(circuit, prefix="SDC1")
steane.append_superdense(circuit, prefix="SDC2")
steane.append_cultivation(circuit, prefix="CULT")

In [ ]:
for color in steane.stabilizers.keys():
    steane.annotate_detector(circuit, f"SDC0:X{color}")
    steane.annotate_detector(circuit, f"SDC0:Z{color}")
    steane.annotate_detector(circuit, f"SDC0:Z{color}", f"SDC1:Z{color}")
    steane.annotate_detector(circuit, f"SDC1:Z{color}", f"SDC2:Z{color}")

for round in range(2):
    steane.annotate_detector(circuit, f"SDC{round+1}:XG", f"SDC{round}:XR", f"SDC{round}:XG")
    steane.annotate_detector(circuit, f"SDC{round+1}:XB", f"SDC{round}:XG")
    steane.annotate_detector(circuit, f"SDC{round+1}:XR")

for measurement in range(6):
    steane.annotate_detector(circuit, f"CULT:X{measurement}")

In [ ]:
print(f"Missing detectors : {len(circuit.missing_detectors())}")

In [ ]:
display(Markdown(f"[Open in Crumble]({circuit.to_crumble_url()})"))

In [ ]:
circuit.to_file("../generated/magic-steane-code.stim")
rewrite_with_polygons("../generated/magic-steane-code.stim", steane)